# 🎬 OpenSource Clipping — One-Click Mode

**Run this single cell.** Just paste a YouTube URL and hit play. It clones/updates the repo, installs deps, runs the pipeline, and prints **everything** in real-time.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ONE CELL — SETUP + RUN + STREAM ALL OUTPUT
# ═══════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil, json, time
from pathlib import Path

# ── Optional: upload cookies.txt for YouTube ──
# If YouTube says "Sign in to confirm you’re not a bot", export cookies from
# your browser while logged in to YouTube and upload them here.
#
# Chrome/Firefox extensions to export cookies.txt:
# • Chrome: "Get cookies.txt LOCALLY"
# • Firefox: "cookies.txt"
#
# Uncomment the next 3 lines and run this cell once, then re-run the whole cell:
# from google.colab import files
# uploaded = files.upload()  # upload cookies.txt
# shutil.move(list(uploaded.keys())[0], "/content/cookies.txt")

# ── Paste YouTube URL below, then run this cell ──
URL_VIDEO = ""  # ← paste YouTube URL here

if not URL_VIDEO.strip():
    raise ValueError("Please paste a YouTube URL in URL_VIDEO above.")

# ── Fixed pipeline settings (edit if you want) ──
SETTINGS = {
    "GOOGLE_API_KEY": "",
    "HF_TOKEN"      : "",
    "PEXELS_API_KEY" : "",
    "NVIDIA_API_KEY" : "",
    "OLLAMA_API_KEY" : "",
    "SOURCE_PLATFORM" : "youtube",
    "SOURCE_HEIGHT" : "max",
    "JUMLAH_CLIP"   : 5,
    "RASIO"         : "9:16",
    "RENDER_HEIGHT" : "1080",
    "AI_PROVIDER"   : "ollama",
    "GEMINI_MODEL"  : "gemini-3-flash-preview",
    "OLLAMA_URL"    : "http://localhost:11434",
    "OLLAMA_MODEL"  : "llama3.1",
    "OLLAMA_FALLBACK_URL"  : "http://localhost:11434",
    "OLLAMA_FALLBACK_MODEL" : "llama3.1",
    "VERBOSE" : False,
    "COOKIES_FILE" : "",  # /content/cookies.txt — helps bypass YouTube bot check
    "USE_YT_TRANSCRIPT_API" : False,
    "USE_DLP_SUBS"          : True,
    "WHISPER_MODEL"         : "large-v3",
    "WHISPER_COMPUTE_TYPE"  : "float16",
    "FONT_STYLE"    : "HORMOZI",
    "FACE_DETECTOR" : "mediapipe",
    "YOLO_SIZE"     : "8m",
    "NO_BGM"     : True,
    "NO_BROLL"   : True,
    "NO_SUBS"    : False,
    "NO_KARAOKE" : False,
    "NO_HOOK"    : False,
    "SPLIT_SCREEN"   : False,
    "SPLIT_TRIGGER"  : "diarization",
    "CAMERA_SWITCH"  : False,
    "HOOK_V2"        : False,
    "NO_SEGMENT_TRIM": False,
    "SILENCE_TRIM"   : False,
    "VIDEO_PRESET"     : "ultrafast",
    "VIDEO_SCALE_ALGO" : "bicubic",
    "VIDEO_CQ"         : 23,
    "VIDEO_CRF"        : 20,
    "VIDEO_SHARPEN"    : False,
    "COLAB_CLEANUP"    : True,
    "WORDS_PER_SUB"    : 5,
    "HOOK_DURATION"    : 3,
}

# ── Repo paths ──
REPO_URL = "https://github.com/troublescope/Ashortai.git"
CLONE_DIR = "/content/opensource-clipping"
Path(CLONE_DIR).mkdir(parents=True, exist_ok=True)

# ── Mount Drive (optional) ──
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# ── Clone / update repo ──
if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=True)
else:
    print("⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

os.chdir(CLONE_DIR)

# ── Install dependencies ──
print("⏳ Installing dependencies…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed.")

# ── Upgrade yt-dlp (helps with YouTube bot checks) ──
print("⏳ Upgrading yt-dlp…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=False)

# ── Restore models from Drive cache if available ──
try:
    DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
    for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
        src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)
            print(f"📥 Restored {m}")
except Exception:
    pass

# ── Write .env file ──
env_lines = [f"{k}={v}" for k, v in SETTINGS.items() if "_API_KEY" in k or k in {"GOOGLE_API_KEY"}]
Path(CLONE_DIR, ".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print("🔐 .env written.")

# ── Build command ──
S = SETTINGS
cmd = [
    sys.executable, "main.py",
    "--url", URL_VIDEO.strip(),
    "--source", S["SOURCE_PLATFORM"],
    "--clips", str(S["JUMLAH_CLIP"]),
    "--ratio", S["RASIO"],
    "--render-height", str(S["RENDER_HEIGHT"]),
    "--source-height", str(S["SOURCE_HEIGHT"]),
    "--font-style", S["FONT_STYLE"],
    "--face-detector", S["FACE_DETECTOR"],
    "--yolo-size", S["YOLO_SIZE"],
    "--ai-provider", S["AI_PROVIDER"],
    "--gemini-model", S["GEMINI_MODEL"],
    "--ollama-url", S["OLLAMA_URL"],
    "--ollama-model", S["OLLAMA_MODEL"],
    "--ollama-fallback-url", S["OLLAMA_FALLBACK_URL"],
    "--ollama-fallback-model", S["OLLAMA_FALLBACK_MODEL"],
    "--whisper-model", S["WHISPER_MODEL"],
    "--whisper-compute-type", S["WHISPER_COMPUTE_TYPE"],
    "--video-preset", S["VIDEO_PRESET"],
    "--video-scale-algo", S["VIDEO_SCALE_ALGO"],
    "--video-cq", str(S["VIDEO_CQ"]),
    "--video-crf", str(S["VIDEO_CRF"]),
    "--words-per-sub", str(S["WORDS_PER_SUB"]),
    "--hook-duration", str(S["HOOK_DURATION"]),
]

def add(flag, key):
    if S.get(key):
        cmd.append(flag)

add("--colab-cleanup", "COLAB_CLEANUP")
add("--use-yt-transcript-api", "USE_YT_TRANSCRIPT_API")
add("--use-dlp-subs", "USE_DLP_SUBS")
add("--no-bgm", "NO_BGM")
add("--no-broll", "NO_BROLL")
add("--no-subs", "NO_SUBS")
add("--no-karaoke", "NO_KARAOKE")
add("--no-hook", "NO_HOOK")
add("--camera-switch", "CAMERA_SWITCH")
add("--hook-v2", "HOOK_V2")
add("--no-segment-trim", "NO_SEGMENT_TRIM")
add("--silence-trim", "SILENCE_TRIM")
add("--video-sharpen", "VIDEO_SHARPEN")
add("--verbose", "VERBOSE")
add("--verbose", "VERBOSE")

if S.get("COOKIES_FILE") and Path(S["COOKIES_FILE"]).exists():
    cmd += ["--cookies", S["COOKIES_FILE"]]

if S.get("SPLIT_SCREEN"):
    cmd.append("--split-screen")
    cmd += ["--split-trigger", S["SPLIT_TRIGGER"]]

# ── Sanity checks ──
if S["OLLAMA_URL"].startswith("https://ollama.com"):
    print("\n⚠️ WARNING: OLLAMA_URL is set to https://ollama.com which is NOT an API endpoint.")
    print("   Change it to http://localhost:11434 or your actual Ollama/OpenAI-compatible API URL.")

# ── Run pipeline and print all output live ──
print("\n🚀 Starting pipeline…")
print("URL:", URL_VIDEO.strip())
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
process = subprocess.Popen(
    cmd,
    cwd=CLONE_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
print("─" * 60)

if process.returncode != 0:
    print(f"❌ Pipeline exited with code {process.returncode}")
else:
    print("✅ Pipeline finished successfully.")

elapsed = time.time() - start
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

# ── Show outputs ──
out_dir = Path(CLONE_DIR) / "outputs"
if out_dir.exists():
    files = sorted(out_dir.glob("*_ready.mp4")) + sorted(out_dir.glob("*.jpg"))
    if files:
        print(f"\n📦 Output files ({len(files)}):")
        for f in files:
            print(f"   • {f.name} ({f.stat().st_size/1024/1024:.1f} MB)")
    if list(out_dir.iterdir()):
        zip_path = Path(CLONE_DIR) / "outputs.zip"
        shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(out_dir))
        print(f"\n🗜️ outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
        try:
            from google.colab import files
            files.download(str(zip_path))
        except Exception:
            pass
else:
    print("⚠️ outputs/ directory not found.")
